# 01 · Exploración — `data/renta_ine`

**Pipeline:** `01_exploracion` → `02_limpieza` → `03_visualizacion`  
**Objetivo:** entender estructura, encoding, separadores, tipos de columna,
NaN patterns y cardinalidades de los 6 ficheros del INE antes de limpiar.

| Fichero | Alias |
|---------|-------|
| `1.Indicadores de renta media y mediana.csv` | `renta` |
| `2.Distribución por fuente de ingresos.csv` | `fuente_ing` |
| `4.Porcentaje de población ... edad.csv` | `umbrales_edad` |
| `5.Porcentaje de población ... nacionalidad.csv` | `umbrales_nacionalid` |
| `9.Índice de Gini y Distribución de la renta P80P20.csv` | `gini` |
| `10.Indicadores demográficos.csv` | `demografico` |


## 0 · Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent   # ajusta si lanzas desde otro dir
DATA_DIR     = PROJECT_ROOT / 'data' / 'renta_ine'

ENCODING  = 'utf-8-sig'
SEPARATOR = ';'
NA_VALUES = ['..', '']

FILES = {
    'renta':               DATA_DIR / '1.Indicadores de renta media y mediana.csv',
    'fuente_ing':          DATA_DIR / '2.Distribución por fuente de ingresos.csv',
    'umbrales_edad':       DATA_DIR / '4.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y tramos de edad.csv',
    'umbrales_nacionalid': DATA_DIR / '5.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y nacionalidad.csv',
    'gini':                DATA_DIR / '9.Índice de Gini y Distribución de la renta P80P20.csv',
    'demografico':         DATA_DIR / '10.Indicadores demográficos.csv',
}

print('DATA_DIR existe:', DATA_DIR.exists())
print('Ficheros encontrados:')
for k, p in FILES.items():
    print(f'  {k:22s}: {p.exists()} | {p.name[:60]}')

## 1 · Vista rápida de cada fichero (head + columnas + shape)

In [ ]:
def peek(alias, path, nrows=3):
    df = pd.read_csv(path, sep=SEPARATOR, encoding=ENCODING,
                     na_values=NA_VALUES, dtype=str, nrows=nrows)
    print(f'\n{'='*70}')
    print(f'  {alias.upper()} — {path.name[:60]}')
    print(f'  Columnas ({len(df.columns)}): {list(df.columns)}')
    display(df)

for alias, path in FILES.items():
    peek(alias, path)

## 2 · Estadísticas de carga — shape, NaN, dtypes

In [ ]:
raw = {}
summary = []

for alias, path in FILES.items():
    df = pd.read_csv(path, sep=SEPARATOR, encoding=ENCODING,
                     na_values=NA_VALUES, dtype=str, low_memory=True)
    raw[alias] = df
    total_cells = df.size
    nan_cells   = df.isna().sum().sum()
    summary.append({
        'alias':   alias,
        'filas':   len(df),
        'cols':    len(df.columns),
        'NaN':     nan_cells,
        'NaN_%':   round(nan_cells / total_cells * 100, 1),
    })

df_summary = pd.DataFrame(summary).set_index('alias')
print('RESUMEN DE CARGA')
display(df_summary)

## 3 · Estructura interna — columnas clave por fichero

Los CSVs del INE siguen el formato *pivotado*:  
- Columnas de **dimensión** (geo + filtros + indicador) → varían por fichero  
- Columna **`Periodo`** → año  
- Columna **`Total`** → valor numérico (con decimal `,`)  

La columna que precede a `Periodo` y `Total` es la que codifica el **indicador**.

In [ ]:
for alias, df in raw.items():
    cols = df.columns.tolist()
    col_ind = cols[-3]   # penúltima antes de Periodo y Total
    print(f'\n── {alias.upper()} ──────────────────────────────────────────')
    print(f'  Todas las columnas  : {cols}')
    print(f'  → col. indicador    : "{col_ind}"')
    print(f'  → col. geo/filtro   : {cols[:-3]}')
    print(f'  Valores únicos [{col_ind}]:')
    for v in sorted(df[col_ind].dropna().unique()):
        print(f'     · {v}')
    # años disponibles
    if 'Periodo' in df.columns:
        years = sorted(df['Periodo'].dropna().unique())
        print(f'  Años disponibles    : {years}')

## 4 · Cardinalidad y valores únicos de columnas geo

In [ ]:
GEO_COLS = ['Municipios', 'Distritos', 'Secciones']

for alias, df in raw.items():
    geo_present = [c for c in GEO_COLS if c in df.columns]
    if not geo_present:
        continue
    print(f'\n── {alias.upper()} ──')
    for gc in geo_present:
        nuniq = df[gc].nunique()
        sample = df[gc].dropna().sample(min(3, nuniq), random_state=42).tolist()
        print(f'  {gc}: {nuniq} únicos  |  sample: {sample}')
    # comprueba si contiene código 5-dígitos o nombre
    gc0 = geo_present[0]
    first_val = str(df[gc0].dropna().iloc[0])
    print(f'  Primer valor geo: "{first_val}"  (len={len(first_val)})')

## 5 · Análisis del campo `Total` — parseado numérico

In [ ]:
for alias, df in raw.items():
    if 'Total' not in df.columns:
        print(f'{alias}: sin columna Total (?)')
        continue
    raw_sample = df['Total'].dropna().head(5).tolist()
    # parseo: reemplaza punto de miles y coma decimal
    s = (df['Total']
           .str.replace('.', '', regex=False)
           .str.replace(',', '.', regex=False)
           .pipe(pd.to_numeric, errors='coerce'))
    print(f'\n── {alias.upper()} ──')
    print(f'  Raw samples     : {raw_sample}')
    print(f'  Parseado  min   : {s.min():.2f}')
    print(f'  Parseado  max   : {s.max():.2f}')
    print(f'  Parseado  NaN%  : {s.isna().mean()*100:.1f}%')

## 6 · NaN patterns — ¿dónde faltan datos?

In [ ]:
for alias, df in raw.items():
    nan_by_col = df.isna().sum()
    nan_by_col = nan_by_col[nan_by_col > 0]
    print(f'\n── {alias.upper()} — columnas con NaN ──')
    if nan_by_col.empty:
        print('  Ninguna columna con NaN ✅')
    else:
        for col, n in nan_by_col.items():
            print(f'  {col:50s}: {n:>7,} NaN  ({n/len(df)*100:.1f}%)')

## 7 · Muestra de valores únicos de filtros (sexo, tramos, nacionalidad)

In [ ]:
FILTER_COLS = ['Sexo', 'Edad', 'Nacionalidad', 'Tipo de dato', 'Indicadores de demografía']

for alias, df in raw.items():
    found = [c for c in FILTER_COLS if c in df.columns]
    if not found:
        continue
    print(f'\n── {alias.upper()} ──')
    for fc in found:
        vals = sorted(df[fc].dropna().unique())
        print(f'  {fc:40s}: {vals}')

## 8 · Resultados de la ejecución por línea de comandos

Los resultados del script de validación ya ejecutado confirman:

| Fichero | Filas | NaN% |
|---------|-------|------|
| `renta` | 427,140 | 13.6% |
| `fuente_ing` | 366,255 | 20.4% |
| `umbrales_edad` | 78,084 | 70.3% |
| `umbrales_nacionalid` | 96,120 | 89.7% |
| `gini` | 146,502 | 19.2% |
| `demografico` | 447,804 | 9.2% |

**Observaciones clave:**
- `umbrales_edad` y `umbrales_nacionalid` tienen NaN muy alto porque incluyen 
  desgloses por sexo/tramo/nación que no aplican a todos los municipios  
- `renta_neta_media` tiene **100% completitud** en todos los años → variable más fiable  
- Años cubiertos: **2015–2023** en todos los ficheros  
- Indicadores demográficos solo alcanzan **92% cobertura** → provincias pequeñas con municipios sin dato  
- **0 anomalías temporales** (variación >±20%) en renta neta ✅  
- Completitud global del dataset provincia×año: **96.1%** ✅

## 9 · Decisiones de diseño → input para `02_ine.ipynb`

### Estrategia de filtrado
- **`renta`**: sin filtros extra (una fila por municipio×año×indicador)  
- **`fuente_ing`**: sin filtros (ya es % del total de ingresos)  
- **`umbrales_edad`**: filtrar `Sexo == 'Total'` + `Edad == 'Total'` → 1 fila por mun×año×umbral  
- **`umbrales_nacionalid`**: filtrar `Sexo == 'Total'` + `Nacionalidad == 'Total'` → 1 fila por mun×año×umbral  
- **`gini`**: sin filtros (solo 2 indicadores)  
- **`demografico`**: sin filtros  

### Parsing numérico
```python
# punto = separador de miles, coma = decimal
s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False).pipe(pd.to_numeric, errors='coerce')
```

### Granularidad objetivo
- **Nivel municipio×año** (`cod_municipio`, `year`) → dataset principal: **63,979 filas × 25 cols**  
- **Nivel provincia×año** (`cod_provincia`, `year`) → dataset agregado: **449 filas × 24 cols**  

### Outputs del pipeline
```
output/clean/
├── ine_municipio_anyo.parquet   ← merge de 4 fuentes × municipio × año
└── ine_provincia_anyo.parquet  ← agregación a provincia × año
```

---
> **Siguiente paso:** `notebooks/02_limpieza/02_ine.ipynb`  
> Aplica los filtros, parsea `Total`, hace el pivot + merge y escribe los `.parquet`.